In [1]:
import pandas as pd
import numpy as np

In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))
from config import raw_data, control_data, external_data, raw_kyiv, filtered_data, daily_data

In [3]:
raw = raw_data

text = pd.read_csv(raw / "text_messages_sorted.csv")
audio = pd.read_csv(raw / "audio_messages_sorted.csv")
posts = pd.read_csv(raw / "posts_sorted.csv")
reactions = pd.read_csv(raw / "reactions_sorted.csv")
comments = pd.read_csv(raw / "comments_sorted.csv")

text["datetime"] = pd.to_datetime(text["datetime"])
audio["datetime"] = pd.to_datetime(audio["datetime"])

In [4]:
group_chats_count = text.groupby("conversation_id")["sender_id"].nunique().gt(2).sum()
n_chats = text["conversation_id"].nunique()

raw_vals = {
    "Donations": text["donation_id"].nunique(),
    "Chats": f"{n_chats} ({round(group_chats_count / n_chats * 100, 2)}% group chats)",
    "Group chats": group_chats_count,
    "Total messages": len(text) + len(audio),
    "Text messages": len(text),
    "Audio messages": len(audio),
    "Posts": len(posts),
    "Reactions": len(reactions),
    "Comments": len(comments),
}

In [5]:
filtered = filtered_data

text = pd.read_csv(filtered / "text_messages_filtered.csv")
audio = pd.read_csv(filtered / "audio_messages_filtered.csv")
posts = pd.read_csv(filtered / "posts_sorted.csv")
reactions = pd.read_csv(filtered / "reactions_sorted.csv")
comments = pd.read_csv(filtered / "comments_sorted.csv")

text["datetime"] = pd.to_datetime(text["datetime"])
audio["datetime"] = pd.to_datetime(audio["datetime"])

In [6]:
group_chats_count = text.groupby("conversation_id")["sender_id"].nunique().gt(2).sum()
n_chats = text["conversation_id"].nunique()

filtered_vals = {
    "Donations": text["donation_id"].nunique(),
    "Chats": f"{n_chats} ({round(group_chats_count / n_chats * 100, 2)}% group chats)",
    "Group chats": group_chats_count,
    "Total messages": len(text) + len(audio),
    "Text messages": len(text),
    "Audio messages": len(audio),
    "Posts": len(posts),
    "Reactions": len(reactions),
    "Comments": len(comments),
}

In [7]:
summary = pd.DataFrame({
    "Donation characteristics": raw_vals.keys(),
    "Raw": raw_vals.values(),
    "Filtered": filtered_vals.values(),
})
summary

,Donation characteristics,Raw,Filtered
0,Donations,24,24
1,Chats,12762 (2.77% group chats),1914 (18.44% group chats)
2,Group chats,353,353
3,Total messages,2478153,2324916
4,Text messages,2375291,2224658
5,Audio messages,102862,100258
6,Posts,438,438
7,Reactions,266048,266048
8,Comments,6366,6366


In [8]:
chats_per_person = text.groupby("donation_id")["conversation_id"].nunique()
group_chats_per_person = (
    text.groupby(["donation_id", "conversation_id"])["sender_id"]
    .nunique()
    .gt(2)
    .groupby(level="donation_id")
    .sum()
)
text_per_person = text.groupby("donation_id").size()
audio_per_person = audio.groupby("donation_id").size()
messages_per_person = text_per_person + audio_per_person.reindex(text_per_person.index, fill_value=0)
reactions_per_person = reactions.groupby("donation_id").size()
comments_per_person = comments.groupby("donation_id").size()
timespan = text.groupby("donation_id")["datetime"].agg(lambda x: (x.max() - x.min()).days)

In [9]:
def stats(series):
    return {
        "Median": round(series.median()),
        "Mean": round(series.mean(), 2),
        "SD": round(series.std(), 2),
        "Range": f"{series.min()}–{series.max()}"
    }

In [10]:
series_list = [
    ("Donation timespan (Days)", timespan),
    ("Chats", chats_per_person),
    ("Group chats", group_chats_per_person),
    ("Messages", messages_per_person),
    ("Text messages", text_per_person),
    ("Audio messages", audio_per_person),
    ("Reactions", reactions_per_person),
    ("Comments", comments_per_person),
]

In [11]:
stats_table = pd.DataFrame([
    {"Metric": name, **stats(series)}
    for name, series in series_list
])
stats_table

,Metric,Median,Mean,SD,Range
0,Donation timespan (Days),1623,1724.75,309.99,1268–2379
1,Chats,60,79.75,59.05,32–320
2,Group chats,10,14.71,13.97,0–60
3,Messages,81244,96871.50,63474.48,17239–304325
4,Text messages,78791,92694.08,60534.14,17029–289772
5,Audio messages,2676,4177.42,4054.23,210–14553
6,Reactions,11147,11085.33,8394.87,717–28937
7,Comments,278,265.25,200.74,20–740
